# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list all record sets and their contained fields (columns), referencing their `@id` for each entity. This is crucial for correct extraction in subsequent sections.

In [ ]:
# Access all record sets in the dataset
all_record_sets = list(dataset.record_sets())
print(f"Number of record sets in dataset: {len(all_record_sets)}\n")

for record_set in all_record_sets:
    rs_id = record_set['@id'] if '@id' in record_set else None
    rs_name = record_set.get('name', '<no name>')
    print(f"RecordSet @id: {rs_id}\n  name: {rs_name}")
    # List the fields (columns)
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields (columns):")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) and '@id' in field else field
        print(f"    - Field @id: {field_id}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Below, we extract all data from each record set by its `@id` into a pandas DataFrame. You can choose the record set you wish to analyze further by its `@id` as shown above.

In [ ]:
# Collect the @id of each record set
record_set_ids = [rs['@id'] for rs in dataset.record_sets()]
print(f"Record set IDs: {record_set_ids}")

# Load each record set into its DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f'Loading records for record set {record_set_id} ...')
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        if not df.empty:
            print(f"Fields: {df.columns.tolist()}")
            dataframes[record_set_id] = df
            print(df.head(2))
    print("")
# Let's choose the first record set with available data for further analysis
chosen_record_set_id = None
if dataframes:
    chosen_record_set_id = list(dataframes.keys())[0]
    print(f"Using record set '{chosen_record_set_id}' for downstream analysis.")
else:
    print("No dataframes loaded. Please check record sets above.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In this example, we'll:
- Select a numeric field for filtering and normalization
- Demonstrate filtering records (e.g., where abundance > threshold)
- Normalize the selected field
- Optionally group by a categorical field

All fields are referenced by their exact `@id` or canonical column name.

In [ ]:
if chosen_record_set_id:
    df = dataframes[chosen_record_set_id]
    print(f"Available columns in selected record set: {df.columns.tolist()}")
    
    # Example: Try to automatically select a numeric field relevant for EDA
    numeric_field = None
    for col in df.columns:
        # Try to guess numeric column by data type
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None:
        # Fallback: Try to find columns with common numeric names
        for nn in ['abundance', 'Abundance', 'MW', 'Coverage', 'Count', 'count', 'Number', 'number', 'value', 'Value']:
            for col in df.columns:
                if nn in col:
                    try:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                        if pd.api.types.is_numeric_dtype(df[col]):
                            numeric_field = col
                            break
                    except Exception:
                        continue
            if numeric_field:
                break
    if numeric_field:
        print(f"Using numeric field '{numeric_field}' for analysis.")
        # Show basic stats to pick a threshold
        print(df[numeric_field].describe())
        # Set a threshold at, for example, mean value
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} rows")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by first non-numeric (likely categorical) column
        group_field = None
        for col in df.columns:
            if col == numeric_field: continue
            if not pd.api.types.is_numeric_dtype(df[col]):
                group_field = col
                break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by '{group_field}', showing mean of '{numeric_field}':")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for filtering/normalization.")
else:
    print("No record set DataFrame selected for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the chosen numeric field, and if grouped data is available, a barplot comparing group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set_id and numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()
    # Bar plot of group means if available
    if 'grouped_df' in locals() and group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()
else:
    print('No suitable numeric field or data to visualize.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded the metadata and records from the dataset using the Croissant schema and `mlcroissant` library.
- Record sets and field `@id`s were listed and all data is referenced accordingly.
- Basic filtering, normalization, grouping, and visualizations were performed for quick assessment.
- For more domain-specific analysis, details regarding biological/experimental meaning of columns can be referenced from the FAIR² metadata and Croissant schema.

Explore the record sets and fields further by adapting the above template and referencing record set and field `@id` values as demonstrated.